In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"hocminhle","key":"29d5a2d7248740a6b9e9ae3220f1db6e"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Download BrATS2020 dataset

In [4]:
!kaggle datasets download -d awsaf49/brats2020-training-data
!unzip brats2020-training-data.zip -d /content/brats2020

Streaming output truncated to the last 5000 lines.
  inflating: /content/brats2020/BraTS2020_training_data/content/data/volume_70_slice_63.h5  
  inflating: /content/brats2020/BraTS2020_training_data/content/data/volume_70_slice_64.h5  
  inflating: /content/brats2020/BraTS2020_training_data/content/data/volume_70_slice_65.h5  
  inflating: /content/brats2020/BraTS2020_training_data/content/data/volume_70_slice_66.h5  
  inflating: /content/brats2020/BraTS2020_training_data/content/data/volume_70_slice_67.h5  
  inflating: /content/brats2020/BraTS2020_training_data/content/data/volume_70_slice_68.h5  
  inflating: /content/brats2020/BraTS2020_training_data/content/data/volume_70_slice_69.h5  
  inflating: /content/brats2020/BraTS2020_training_data/content/data/volume_70_slice_7.h5  
  inflating: /content/brats2020/BraTS2020_training_data/content/data/volume_70_slice_70.h5  
  inflating: /content/brats2020/BraTS2020_training_data/content/data/volume_70_slice_71.h5  
  inflating: /conten

Check dataset folder

In [5]:
import os

for root, dirs, files in os.walk('/content/brats2020'):
    print(root, len(files))
    if len(files) > 0:
        print(files[:5])
        break

/content/brats2020 1
['BraTS20 Training Metadata.csv']


Load .h5 files and create labels

In [6]:
import os
import h5py
import numpy as np
from tqdm import tqdm

data_path = "/content/brats2020/BraTS2020_training_data/content/data"

all_files = sorted([f for f in os.listdir(data_path) if f.endswith(".h5")])

print("Total files:", len(all_files))
print(all_files[:5])

Total files: 57195
['volume_100_slice_0.h5', 'volume_100_slice_1.h5', 'volume_100_slice_10.h5', 'volume_100_slice_100.h5', 'volume_100_slice_101.h5']


In [ ]:
Load .h5 files and create labels

In [7]:
import os
import h5py
import numpy as np
from tqdm import tqdm

data_path = "/content/brats2020/BraTS2020_training_data/content/data"

all_files = sorted([f for f in os.listdir(data_path) if f.endswith(".h5")])

print("Total files:", len(all_files))
print("First files:", all_files[:5])

X = []
y = []

MAX_SAMPLES = 1000

for file in tqdm(all_files[:MAX_SAMPLES]):
    file_path = os.path.join(data_path, file)

    with h5py.File(file_path, "r") as f:
        image = np.array(f["image"])
        mask = np.array(f["mask"])

    label = 1 if np.max(mask) > 0 else 0

    X.append(image)
    y.append(label)

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Tumor:", np.sum(y == 1))
print("No tumor:", np.sum(y == 0))

Total files: 57195
First files: ['volume_100_slice_0.h5', 'volume_100_slice_1.h5', 'volume_100_slice_10.h5', 'volume_100_slice_100.h5', 'volume_100_slice_101.h5']


100%|██████████| 1000/1000 [00:11<00:00, 86.89it/s]


X shape: (1000, 240, 240, 4)
y shape: (1000,)
Tumor: 388
No tumor: 612


Preprocessing

In [8]:
import tensorflow as tf

X = X.astype("float32")

# Normalize
X = X / np.max(X)

# Nếu ảnh có nhiều modality/channel, chỉ lấy 3 channel đầu để dùng MobileNetV2
if X.shape[-1] > 3:
    X = X[:, :, :, :3]

# Nếu ảnh chỉ có 1 channel thì repeat thành 3 channel
if X.shape[-1] == 1:
    X = np.repeat(X, 3, axis=-1)

# Resize về 224x224
X_resized = tf.image.resize(X, (224, 224)).numpy()

print("Final X shape:", X_resized.shape)

Final X shape: (1000, 224, 224, 3)


Train/test split

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_resized,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(800, 224, 224, 3) (200, 224, 224, 3)


Build transfer learning model

In [10]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers
import tensorflow as tf

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
embedding = layers.Dense(128, activation="relu", name="embedding")(x)
x = layers.Dropout(0.3)(embedding)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Dense)               │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Train classification head

In [11]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=32
)

Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 26s 342ms/step - accuracy: 0.6578 - loss: 0.6390 - val_accuracy: 0.7250 - val_loss: 0.4942
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.7797 - loss: 0.4477 - val_accuracy: 0.7750 - val_loss: 0.4391
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.8094 - loss: 0.3821 - val_accuracy: 0.8188 - val_loss: 0.3845
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.8438 - loss: 0.3464 - val_accuracy: 0.8313 - val_loss: 0.3618
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.8516 - loss: 0.3116 - val_accuracy: 0.8375 - val_loss: 0.3389


Fine-tune last layers

In [12]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_finetune = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=32
)

Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 23s 364ms/step - accuracy: 0.6266 - loss: 1.7178 - val_accuracy: 0.8375 - val_loss: 0.3625
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.7906 - loss: 0.6200 - val_accuracy: 0.8188 - val_loss: 0.3999
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.8328 - loss: 0.4535 - val_accuracy: 0.8062 - val_loss: 0.3960
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.8609 - loss: 0.3145 - val_accuracy: 0.8125 - val_loss: 0.3814
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.8656 - loss: 0.2856 - val_accuracy: 0.8125 - val_loss: 0.3802


Evaluate CNN model

In [13]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).ravel()

print("CNN Classification Report")
print(classification_report(y_test, y_pred))

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

cnn_acc = accuracy_score(y_test, y_pred)
cnn_precision = precision_score(y_test, y_pred)
cnn_recall = recall_score(y_test, y_pred)
cnn_f1 = f1_score(y_test, y_pred)

print("CNN Accuracy:", cnn_acc)
print("CNN Precision:", cnn_precision)
print("CNN Recall:", cnn_recall)
print("CNN F1:", cnn_f1)

7/7 ━━━━━━━━━━━━━━━━━━━━ 25s 3s/step
CNN Classification Report
              precision    recall  f1-score   support

           0       0.98      0.70      0.82       122
           1       0.68      0.97      0.80        78

    accuracy                           0.81       200
   macro avg       0.83      0.84      0.81       200
weighted avg       0.86      0.81      0.81       200

Confusion Matrix
[[86 36]
 [ 2 76]]
CNN Accuracy: 0.81
CNN Precision: 0.6785714285714286
CNN Recall: 0.9743589743589743
CNN F1: 0.8


Extract embeddings from trained model

In [14]:
# Call model 1 time build graph
_ = model.predict(X_train[:1])

embedding_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("embedding").output
)

train_embeddings = embedding_model.predict(X_train)
test_embeddings = embedding_model.predict(X_test)

print("Train embeddings:", train_embeddings.shape)
print("Test embeddings:", test_embeddings.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 880ms/step
Train embeddings: (800, 128)
Test embeddings: (200, 128)


Apply PCA dimensionality reduction

In [15]:
from sklearn.decomposition import PCA

pca = PCA(n_components=50)

X_train_pca = pca.fit_transform(train_embeddings)
X_test_pca = pca.transform(test_embeddings)

print("Before PCA:", train_embeddings.shape)
print("After PCA:", X_train_pca.shape)

print("Explained variance ratio:", np.sum(pca.explained_variance_ratio_))

Before PCA: (800, 128)
After PCA: (800, 50)
Explained variance ratio: 0.9994782


Train SVM after PCA

In [16]:
from sklearn.svm import SVC

svm_model = SVC(kernel="rbf", probability=True)

svm_model.fit(X_train_pca, y_train)

y_pred_pca = svm_model.predict(X_test_pca)

print("PCA + SVM Classification Report")
print(classification_report(y_test, y_pred_pca))

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred_pca))

pca_acc = accuracy_score(y_test, y_pred_pca)
pca_precision = precision_score(y_test, y_pred_pca)
pca_recall = recall_score(y_test, y_pred_pca)
pca_f1 = f1_score(y_test, y_pred_pca)

print("PCA + SVM Accuracy:", pca_acc)
print("PCA + SVM Precision:", pca_precision)
print("PCA + SVM Recall:", pca_recall)
print("PCA + SVM F1:", pca_f1)

PCA + SVM Classification Report
              precision    recall  f1-score   support

           0       0.92      0.84      0.88       122
           1       0.78      0.88      0.83        78

    accuracy                           0.85       200
   macro avg       0.85      0.86      0.85       200
weighted avg       0.86      0.85      0.86       200

Confusion Matrix
[[102  20]
 [  9  69]]
PCA + SVM Accuracy: 0.855
PCA + SVM Precision: 0.7752808988764045
PCA + SVM Recall: 0.8846153846153846
PCA + SVM F1: 0.8263473053892215


Compare results

In [17]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Fine-tuned MobileNetV2",
        "MobileNetV2 Embedding + PCA + SVM"
    ],
    "Accuracy": [
        cnn_acc,
        pca_acc
    ],
    "Precision": [
        cnn_precision,
        pca_precision
    ],
    "Recall": [
        cnn_recall,
        pca_recall
    ],
    "F1-score": [
        cnn_f1,
        pca_f1
    ]
})

results

,Model,Accuracy,Precision,Recall,F1-score
0,Fine-tuned MobileNetV2,0.810,0.678571,0.974359,0.800000
1,MobileNetV2 Embedding + PCA + SVM,0.855,0.775281,0.884615,0.826347


Save model and results to Google Drive

In [18]:
save_path = "/content/drive/MyDrive/Brain_Tumor_Project"

os.makedirs(save_path, exist_ok=True)

model.save(os.path.join(save_path, "mobilenetv2_brain_tumor_model.h5"))
results.to_csv(os.path.join(save_path, "model_results.csv"), index=False)

print("Saved to:", save_path)

Saved to: /content/drive/MyDrive/Brain_Tumor_Project
